# Đồ án Trực quan hóa Dữ liệu: Climate Lab
## Quy trình Làm sạch Dữ liệu & Khám phá EDA — Nhiệt độ Toàn cầu và Quốc gia
**Thành viên phụ trách:** Huỳnh Cao Trung Đức (Nhóm 18)

Notebook này thực hiện trọn vẹn luồng dữ liệu nhiệt độ:
1. **Làm sạch chuỗi nhiệt độ toàn cầu** từ NASA GISTEMP v4 (1880 – 2025).
2. **Làm sạch & chuẩn hóa chuỗi nhiệt độ quốc gia** từ FAOSTAT (1961 – 2025) bằng chuẩn **UN M49 numeric -> ISO3**, tách bạch chính xác Trung Quốc (`CHN`) khỏi vùng tổng hợp và tách riêng các đặc khu (`HKG`, `MAC`, `TWN`).
3. **Báo cáo kiểm định chất lượng dữ liệu** (Data Quality Audit): Data dictionary, kiểm tra số dòng trước/sau, missing, outlier, các mã không ghép và xuất file `bao_cao_chat_luong.json`.
4. **Khám phá dữ liệu (EDA)**: Notebook xuất 5 biểu đồ tĩnh. File `tao_bieu_do_plotly.py` tạo 6 biểu đồ tương tác: xu hướng theo năm, trung bình thập kỷ, bản đồ quốc gia, heatmap châu lục, boxplot và chênh nhiệt độ theo tháng. Dashboard áp dụng bộ lọc năm, châu lục và quốc gia. Phần hồi quy tuyến tính được trình bày riêng tại `mo_hinh_du_doan/nguyen_khang`.

### Bước 1: Khai báo thư viện và cấu hình thư mục làm việc

In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px


if Path('phan_tich_nhiet_do/duc').is_dir():
    os.chdir('phan_tich_nhiet_do/duc')
from lam_sach_du_lieu import clean_nasa, clean_faostat, clean_monthly, export_quality_report

plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False
for folder in ['bieu_do/tinh', 'bieu_do/tuong_tac']:
    Path(folder).mkdir(parents=True, exist_ok=True)


### Bước 2: Làm sạch dữ liệu nhiệt độ toàn cầu NASA GISTEMP v4 (1880 – 2025)
- Đọc file thô `nasa_nhiet_do_toan_cau_1880_2026.csv`.
- Trích xuất cột `Year` và cột `J-D` (độ lệch nhiệt độ trung bình năm toàn cầu so với thời kỳ cơ sở 1951–1980).
- Loại bỏ năm 2026 do chưa đủ 12 tháng.
- Bổ sung cột thập kỷ `decade` và xuất ra `data/du_lieu_da_xu_ly/duc/nhiet_do_toan_cau.csv`.

In [ ]:
df_nasa_final, nasa_stats = clean_nasa()
display(df_nasa_final.head(10))


### Bước 3: Làm sạch & Chuẩn hóa nhiệt độ quốc gia FAOSTAT (1961 – 2025) bằng chuẩn UN M49 $\rightarrow$ ISO3

> **Quy tắc làm sạch bắt buộc:**
> 1. **Dùng cột `Area Code (M49)`**: Thay vì so khớp chuỗi tên `Area` (dễ lỗi), sử dụng trực tiếp mã số UN M49 numeric để đối chiếu sang ISO-3166-1 alpha-3.
> 2. **Khắc phục lỗi trường hợp Trung Quốc**:
>    - Mã M49 `'156'` (`China, mainland`) $\rightarrow$ gán mã ISO3 chuẩn **`CHN`**, tên chuẩn hóa **`China`**.
>    - Các vùng hành chính đặc quyền và lãnh thổ: `'344'` $\rightarrow$ **`HKG`**, `'446'` $\rightarrow$ **`MAC`**, `'158'` $\rightarrow$ **`TWN`**.
>    - Mã M49 `'159'` (`China` tổng hợp trong FAO) là mã nhóm của FAO $\rightarrow$ tách riêng vào danh sách **vùng tổng hợp (aggregates)** cùng với `World ('001')`, `Asia ('142')`, `Europe ('150')`... để tránh cộng trùng số liệu.
> 3. **Gắn Châu lục**: Dùng danh mục chuẩn của OWID để gán châu lục đồng bộ cho toàn bộ dự án.

In [ ]:
df_fao_final, fao_stats = clean_faostat()
df_monthly = clean_monthly()
display(df_fao_final.head(10))
display(df_fao_final[df_fao_final.iso_alpha.isin(['CHN', 'HKG', 'MAC', 'TWN'])]
        [['country', 'iso_alpha', 'continent']].drop_duplicates())


### Bước 4: Báo cáo Kiểm tra Chất lượng Dữ liệu (Data Quality Audit) & Data Dictionary
Báo cáo này tổng hợp đầy đủ theo chuẩn khoa học:
1. **Data Dictionary**: Ý nghĩa, kiểu dữ liệu, phạm vi và mốc quy chiếu của từng trường.
2. **Số dòng trước và sau làm sạch** của cả NASA và FAOSTAT.
3. **Tỷ lệ giá trị thiếu (Missing values)** trên từng cột.
4. **Danh sách các vùng tổng hợp (Aggregates)** được tách riêng để không tính trùng.
5. **Phân tích phân phối và Outliers** (IQR: bóc tách rõ ràng ngoại lai nóng và ngoại lai lạnh, kiểm tra cờ dữ liệu).

In [ ]:
export_quality_report(nasa_stats, fao_stats)
display(pd.DataFrame({'NASA': nasa_stats['missing_rate'],
                      'FAOSTAT': fao_stats['missing_rate']}))
print('Đã lưu data/du_lieu_da_xu_ly/duc/bao_cao_chat_luong.json.')


### Bước 5: Trực quan hóa Dữ liệu Khám phá (EDA)
#### 1. Line Chart (Xu hướng nhiệt độ toàn cầu) & 2. Bar Chart (Nhiệt độ theo thập kỷ)

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df_nasa_final['year'], df_nasa_final['temperature_anomaly'], color='#f46d43', alpha=0.5, label='Nhiệt độ hàng năm')
df_nasa_final['rolling_5'] = df_nasa_final['temperature_anomaly'].rolling(5).mean()
plt.plot(df_nasa_final['year'], df_nasa_final['rolling_5'], color='#a50026', linewidth=2.5, label='Đường xu hướng (Trung bình 5 năm)')
plt.axhline(0, color='blue', linestyle='--', linewidth=1, label='Baseline 1951-1980 (0°C)')
plt.title('Xu hướng Nhiệt độ Toàn cầu qua Thời gian (1880 – 2025)', fontsize=14, fontweight='bold')
plt.xlabel('Năm')
plt.ylabel('Độ lệch nhiệt độ (°C)')
plt.legend()
plt.tight_layout()
plt.savefig('bieu_do/tinh/01_xu_huong_nhiet_do_toan_cau.png', dpi=300)
plt.show()


df_dec = df_nasa_final.groupby('decade')['temperature_anomaly'].mean().reset_index()
colors = ['#4575b4' if v < 0 else '#d73027' for v in df_dec['temperature_anomaly']]
plt.figure(figsize=(11, 5))
plt.bar(df_dec['decade'].astype(str) + 's', df_dec['temperature_anomaly'], color=colors, width=0.6)
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Nhiệt độ Trung bình Toàn cầu theo Thập kỷ (NASA GISS)', fontsize=14, fontweight='bold')
plt.xlabel('Thập kỷ')
plt.ylabel('Độ lệch (°C)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('bieu_do/tinh/02_nhiet_do_theo_thap_ky.png', dpi=300)
plt.show()

#### 3. Heatmap (Châu lục x Thập kỷ) & 4. Boxplot (Phân bố nhiệt độ quốc gia)
> **Lưu ý chuẩn thị giác:** Thang màu Heatmap được đặt **tâm tại đúng 0.00°C** (`center=0`), giúp phân định rõ ràng giữa các thập kỷ mang nhiệt độ âm (xanh) và nhiệt độ dương (đỏ).

In [ ]:
pivot_cont = df_fao_final.pivot_table(index='continent', columns='decade', values='temperature_anomaly', aggfunc='mean')
pivot_cont.columns = [f'{c}s' for c in pivot_cont.columns]
plt.figure(figsize=(11, 5))
sns.heatmap(
    pivot_cont, cmap='coolwarm', center=0, annot=True, fmt='.2f', linewidths=0.5,
    cbar_kws={'label': 'Độ lệch nhiệt độ (°C)'}
)
plt.title('Nhiệt độ Trung bình theo Châu lục và Thập kỷ (FAOSTAT) [Tâm 0°C]', fontsize=14, fontweight='bold')
plt.xlabel('Thập kỷ')
plt.ylabel('Châu lục')
plt.tight_layout()
plt.savefig('bieu_do/tinh/04_heatmap_chau_luc_thap_ky.png', dpi=300)
plt.show()


plt.figure(figsize=(12, 5))
df_fao_final['decade_str'] = df_fao_final['decade'].astype(str) + 's'
sns.boxplot(data=df_fao_final, x='decade_str', y='temperature_anomaly', hue='decade_str', palette='Reds', showmeans=True, legend=False)
plt.axhline(0, color='blue', linestyle='--', linewidth=0.8, label='Baseline 1951–1980 (0°C)')
plt.title('Phân bố Độ lệch Nhiệt độ giữa các Quốc gia qua từng Thập kỷ (Boxplot)', fontsize=14, fontweight='bold')
plt.xlabel('Thập kỷ')
plt.ylabel('Độ lệch (°C)')
plt.legend(loc='upper left')
plt.tight_layout()
plt.savefig('bieu_do/tinh/05_phan_bo_nhiet_do_quoc_gia.png', dpi=300)
plt.show()

#### 5. Bản đồ chênh nhiệt độ năm 2024
Thang màu cố định từ −6 đến +6 °C: xanh là thấp hơn, trắng là gần bằng và đỏ là cao hơn mức trung bình 1951–1980. Notebook xuất ảnh tĩnh; Dashboard dùng bộ lọc để chọn năm và phạm vi.

In [ ]:
df_2024 = df_fao_final[df_fao_final['year'] == 2024].dropna(subset=['temperature_anomaly'])
fig_static = px.choropleth(
    df_2024,
    locations='iso_alpha',
    color='temperature_anomaly',
    hover_name='country',
    color_continuous_scale='RdBu_r',
    color_continuous_midpoint=0,
    range_color=[-6, 6],
    title='Bản đồ Độ lệch Nhiệt độ Thế giới năm 2024 (so với 1951–1980) [Tâm 0°C]',
    labels={'temperature_anomaly': 'Độ lệch (°C)'},
    template='plotly_white'
)
fig_static.write_image('bieu_do/tinh/03_ban_do_nhiet_do.png', scale=2)